In [15]:
import pandas as pd

# 1. Load Raw Data
df_raw = pd.read_csv('vr_hardware_master_dataset.csv')

# 2. Define standard aggregations (Excluding Best/WorstFrame, Battery Artifacts, CpuClock)
base_metrics = [
    'TotalUsedMem', 'CpuUtil', 'GpuUtil', 
    'BatteryMicroAmps', 'BatteryVoltageMv', 
    'AvgFPS', 'MainThreadMs'
]

# Apply the standard 5 mathematical functions to the base metrics
agg_funcs = {m: ['mean', 'std', 'max', 'min', 'median'] for m in base_metrics}

# 3. Add custom targeted aggregation for Jitter (Preventing bloat)
# We only care about the average jitter, and the single worst jitter spike
agg_funcs['FrameTimeStdDev'] = ['mean', 'max']

# 4. Group and Aggregate
grouped = df_raw.groupby(['db_id', 'room_label', 'noise_type', 'location'])
df_clean = grouped.agg(agg_funcs).reset_index()

# 5. Flatten the column names (e.g., 'TotalUsedMem', 'mean' -> 'TotalUsedMem_mean')
df_clean.columns = ['_'.join(col).strip('_') for col in df_clean.columns.values]

print(f"Original Raw Rows: {len(df_raw)}")
print(f"Aggregated Trials: {len(df_clean)}")
print(f"Total ML Features: {len(df_clean.columns) - 4}") # Ignoring the 4 ID columns

# Save the perfectly clean, un-bloated dataset
df_clean.to_csv('ml_ready_features_v2_optimized.csv', index=False)
print("\nSaved new dataset to 'ml_ready_features_v2_optimized.csv'")

Original Raw Rows: 10087
Aggregated Trials: 109
Total ML Features: 37

Saved new dataset to 'ml_ready_features_v2_optimized.csv'


In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Load the newly optimized dataset
df_clean = pd.read_csv('ml_ready_features_v2_optimized.csv')

# 2. Setup Features and Targets
# Drop the metadata columns to isolate the 37 numeric features
feature_cols = [c for c in df_clean.columns if c not in ['db_id', 'room_label', 'noise_type', 'location']]
X_all = df_clean[feature_cols]

# Force labels and locations to strings to prevent XGBoost float crashes
y_str = df_clean['room_label'].astype(str)
locations = df_clean['location'].astype(str)

# Encode targets to integers for XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y_str)

# ==========================================
# STAGE 1: Recursive Feature Elimination
# ==========================================
print("Starting Recursive Feature Elimination (Targeting Top 15 Features)...\n")

# We use Random Forest as the RFE estimator because it handles multi-class feature importance exceptionally well
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf_base, n_features_to_select=15, step=1)
rfe.fit(X_all, y_encoded)

# Extract the winning features
top_features = X_all.columns[rfe.support_].tolist()

print("The New Top 15 Architectural Features:")
for feat in top_features:
    print(f" - {feat}")

# ==========================================
# STAGE 2: XGBoost Leave-One-Location-Out
# ==========================================
# Isolate X to ONLY use the new Top 15 features
X_selected = df_clean[top_features]

unique_locations = locations.unique()
all_y_true, all_y_pred = [], []

print(f"\nStarting XGBoost LOLO Validation for {len(unique_locations)} locations...\n")

for loc in unique_locations:
    # Safely create numpy boolean masks to prevent Pandas index mismatches
    train_mask = (locations != loc).values
    test_mask = (locations == loc).values
    
    # Apply the masks
    X_train, y_train = X_selected.iloc[train_mask], y_encoded[train_mask]
    X_test, y_test = X_selected.iloc[test_mask], y_encoded[test_mask]
    
    # Initialize and Train XGBoost
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train, y_train)
    
    # Predict
    y_pred = xgb.predict(X_test)
    
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

# ==========================================
# STAGE 3: Final Reporting
# ==========================================
# Force arrays to integers, then reverse transform to strings for the report
all_y_true_int = np.array(all_y_true).astype(int)
all_y_pred_int = np.array(all_y_pred).astype(int)

y_true_labels = le.inverse_transform(all_y_true_int).astype(str)
y_pred_labels = le.inverse_transform(all_y_pred_int).astype(str)

print("--- Optimized XGBoost LOLO Classification Report ---")
print(classification_report(y_true_labels, y_pred_labels))

Starting Recursive Feature Elimination (Targeting Top 15 Features)...

The New Top 15 Architectural Features:
 - TotalUsedMem_mean
 - TotalUsedMem_std
 - TotalUsedMem_max
 - TotalUsedMem_min
 - TotalUsedMem_median
 - CpuUtil_median
 - GpuUtil_mean
 - GpuUtil_max
 - GpuUtil_min
 - GpuUtil_median
 - BatteryMicroAmps_mean
 - BatteryVoltageMv_mean
 - BatteryVoltageMv_max
 - BatteryVoltageMv_min
 - BatteryVoltageMv_median

Starting XGBoost LOLO Validation for 20 locations...

--- Optimized XGBoost LOLO Classification Report ---
              precision    recall  f1-score   support

  conference       0.52      0.40      0.45        30
     hallway       0.78      0.96      0.86        26
     kitchen       0.16      0.15      0.16        26
         lab       0.45      0.48      0.46        27

    accuracy                           0.50       109
   macro avg       0.48      0.50      0.48       109
weighted avg       0.48      0.50      0.48       109



In [17]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Load the optimized dataset
df = pd.read_csv('ml_ready_features_v2_optimized.csv')

# 2. Drop db_id permanently from the dataframe
if 'db_id' in df.columns:
    df = df.drop(columns=['db_id'])
    
# Save this final pristine dataset
df.to_csv('ml_ready_features_final.csv', index=False)
print("Saved pristine dataset (without db_id) to 'ml_ready_features_final.csv'")

# 3. Setup Features and Targets
# We explicitly exclude 'room_label', 'noise_type', and 'location' from the training features (X)
feature_cols = [c for c in df.columns if c not in ['room_label', 'noise_type', 'location']]
X_all = df[feature_cols]

# Isolate the strings to prevent XGBoost float crashes
y_str = df['room_label'].astype(str)
locations = df['location'].astype(str)

le = LabelEncoder()
y_encoded = le.fit_transform(y_str)

# ==========================================
# STAGE 1: Recursive Feature Elimination
# ==========================================
print("\nStarting Recursive Feature Elimination (Targeting Top 15 Features)...")
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf_base, n_features_to_select=15, step=1)
rfe.fit(X_all, y_encoded)

top_features = X_all.columns[rfe.support_].tolist()

print("\nThe New Top 15 Architectural Features:")
for feat in top_features:
    print(f" - {feat}")

# ==========================================
# STAGE 2: XGBoost Leave-One-Location-Out
# ==========================================
X_selected = df[top_features]

unique_locations = locations.unique()
all_y_true, all_y_pred = [], []

print(f"\nStarting XGBoost LOLO Validation for {len(unique_locations)} locations...\n")

for loc in unique_locations:
    # Use explicit masks to prevent index drifting
    train_mask = (locations != loc).values
    test_mask = (locations == loc).values
    
    X_train, y_train = X_selected.iloc[train_mask], y_encoded[train_mask]
    X_test, y_test = X_selected.iloc[test_mask], y_encoded[test_mask]
    
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train, y_train)
    
    y_pred = xgb.predict(X_test)
    
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

# ==========================================
# STAGE 3: Final Reporting
# ==========================================
all_y_true_int = np.array(all_y_true).astype(int)
all_y_pred_int = np.array(all_y_pred).astype(int)

y_true_labels = le.inverse_transform(all_y_true_int).astype(str)
y_pred_labels = le.inverse_transform(all_y_pred_int).astype(str)

print("--- Optimized XGBoost LOLO Classification Report ---")
print(classification_report(y_true_labels, y_pred_labels))

Saved pristine dataset (without db_id) to 'ml_ready_features_final.csv'

Starting Recursive Feature Elimination (Targeting Top 15 Features)...

The New Top 15 Architectural Features:
 - TotalUsedMem_mean
 - TotalUsedMem_std
 - TotalUsedMem_max
 - TotalUsedMem_min
 - TotalUsedMem_median
 - CpuUtil_median
 - GpuUtil_mean
 - GpuUtil_max
 - GpuUtil_min
 - GpuUtil_median
 - BatteryMicroAmps_mean
 - BatteryVoltageMv_mean
 - BatteryVoltageMv_max
 - BatteryVoltageMv_min
 - BatteryVoltageMv_median

Starting XGBoost LOLO Validation for 20 locations...

--- Optimized XGBoost LOLO Classification Report ---
              precision    recall  f1-score   support

  conference       0.52      0.40      0.45        30
     hallway       0.78      0.96      0.86        26
     kitchen       0.16      0.15      0.16        26
         lab       0.45      0.48      0.46        27

    accuracy                           0.50       109
   macro avg       0.48      0.50      0.48       109
weighted avg      

In [18]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. Load the pristine dataset
df = pd.read_csv('ml_ready_features_final.csv')

# 2. Setup Features and Targets
feature_cols = [c for c in df.columns if c not in ['room_label', 'noise_type', 'location']]
X = df[feature_cols]

y_str = df['room_label'].astype(str)
locations = df['location'].astype(str)

le = LabelEncoder()
y_encoded = le.fit_transform(y_str)

# Calculate class weights to force the model to care about Kitchens
# The rarer/harder the class, the higher the weight
from sklearn.utils.class_weight import compute_sample_weight

unique_locations = locations.unique()
all_y_true, all_y_pred = [], []

# Initialize SMOTE (k_neighbors=2 to handle small sample sizes per room)
smote = SMOTE(k_neighbors=2, random_state=42)

print(f"Starting SMOTE + XGBoost LOLO Validation for {len(unique_locations)} locations...\n")

for loc in unique_locations:
    train_mask = (locations != loc).values
    test_mask = (locations == loc).values
    
    X_train, y_train = X.iloc[train_mask], y_encoded[train_mask]
    X_test, y_test = X.iloc[test_mask], y_encoded[test_mask]
    
    # Apply SMOTE mathematically to hallucinate new Kitchens/Labs
    try:
        X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    except ValueError:
        # Fallback if a room class has too few samples after leaving a location out
        X_train_smote, y_train_smote = X_train, y_train
        
    # Calculate dynamic sample weights for the synthesized training data
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_smote)
    
    # Train XGBoost with the new synthetic data AND the sample weights
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train_smote, y_train_smote, sample_weight=sample_weights)
    
    # Predict strictly on the REAL, untouched test data
    y_pred = xgb.predict(X_test)
    
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

# Safe Reporting
all_y_true_int = np.array(all_y_true).astype(int)
all_y_pred_int = np.array(all_y_pred).astype(int)

y_true_labels = le.inverse_transform(all_y_true_int).astype(str)
y_pred_labels = le.inverse_transform(all_y_pred_int).astype(str)

print("--- SMOTE + Weighted XGBoost LOLO Classification Report ---")
print(classification_report(y_true_labels, y_pred_labels))

Starting SMOTE + XGBoost LOLO Validation for 20 locations...

--- SMOTE + Weighted XGBoost LOLO Classification Report ---
              precision    recall  f1-score   support

  conference       0.48      0.37      0.42        30
     hallway       0.77      0.88      0.82        26
     kitchen       0.16      0.15      0.16        26
         lab       0.35      0.41      0.38        27

    accuracy                           0.45       109
   macro avg       0.44      0.45      0.44       109
weighted avg       0.44      0.45      0.44       109



In [20]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 1. Load the pristine, optimized dataset
df = pd.read_csv('ml_ready_features_final.csv')

# 2. Setup Features and Targets
feature_cols = [c for c in df.columns if c not in ['room_label', 'noise_type', 'location']]
X = df[feature_cols]

y_str = df['room_label'].astype(str)
locations = df['location'].astype(str)

le = LabelEncoder()
y_encoded = le.fit_transform(y_str)

# 3. Simulate Minimal Reconnaissance (Pick exactly 1 location per room type)
# We group by room_label and pick the first unique location we see for each
recon_locations = df.groupby('room_label')['location'].first().values

print("--- Attacker's Reconnaissance Footprint (Training Data) ---")
for loc in recon_locations:
    room_type = df[df['location'] == loc]['room_label'].iloc[0]
    print(f" - Scanned: {loc} ({room_type})")

# 4. Split the Data
# Train ONLY on those 4 locations
train_mask = locations.isin(recon_locations).values
# Test on the remaining unseen locations
test_mask = (~locations.isin(recon_locations)).values

X_train, y_train = X.iloc[train_mask], y_encoded[train_mask]
X_test, y_test = X.iloc[test_mask], y_encoded[test_mask]

print(f"\nTraining on {len(X_train)} trials...")
print(f"Attacking {len(X_test)} unseen trials...\n")

# 5. Train the Minimal XGBoost Model
xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train)

# 6. Predict on the unseen building
y_pred = xgb.predict(X_test)

# 7. Generate the Report
y_true_labels = le.inverse_transform(y_test).astype(str)
y_pred_labels = le.inverse_transform(y_pred).astype(str)

print("--- Minimal Reconnaissance XGBoost Classification Report ---")
print(classification_report(y_true_labels, y_pred_labels))

--- Attacker's Reconnaissance Footprint (Training Data) ---
 - Scanned: 3310 (conference)
 - Scanned: Floor4HallwayInFrontOf4214 (hallway)
 - Scanned: Floor3Kitchen (kitchen)
 - Scanned: privateeyelab (lab)

Training on 26 trials...
Attacking 83 unseen trials...

--- Minimal Reconnaissance XGBoost Classification Report ---
              precision    recall  f1-score   support

  conference       0.13      0.08      0.10        24
     hallway       0.53      0.95      0.68        20
     kitchen       0.24      0.28      0.26        18
         lab       0.36      0.19      0.25        21

    accuracy                           0.36        83
   macro avg       0.32      0.38      0.32        83
weighted avg       0.31      0.36      0.31        83

